# BFCL compositional pilot audit

Inspect persistent candidate, guard, pseudo-label, training, and evaluation artifacts after the pilot runs. Set `BFCL_PILOT_RUN_ROOT` to select a run; otherwise the newest timestamped run is used.

In [ ]:
import json, os
from pathlib import Path
import pandas as pd
from IPython.display import display

repo = Path.cwd()
if not (repo / 'artifacts').exists():
    repo = Path('../..').resolve()
configured = os.environ.get('BFCL_PILOT_RUN_ROOT')
runs = sorted((repo / 'artifacts/runs').glob('bfcl_compositional_pilot_*'))
run_root = Path(configured).resolve() if configured else runs[-1]
run_root

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text())

def read_jsonl(path):
    with Path(path).open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

display(read_json(run_root / 'data/audit.json'))
display(read_json(run_root / 'manifest.json'))

## Composed public data and isolated oracle records

In [ ]:
public = read_jsonl(run_root / 'data/public_candidates/round_01_cross.jsonl')
oracle = {row['candidate_id']: row for row in read_jsonl(run_root / 'data/oracle/round_01_cross.jsonl')}
sample = public[0]
assert not {'target', 'canonical_calls', 'accepted_calls', 'evaluator'} & set(sample)
display({k: sample[k] for k in ['candidate_id', 'template_id', 'source_component_ids', 'question', 'functions']})
display(oracle[sample['candidate_id']])

## Guard precision and rejection reasons

In [ ]:
summaries = []
for condition_dir in sorted((run_root / 'round_01/conditions').glob('*')):
    path = condition_dir / 'pseudo_label_audit/summary.json'
    if path.exists():
        summaries.append({'condition': condition_dir.name, **read_json(path)})
pd.DataFrame(summaries).set_index('condition')

In [ ]:
condition = 'compose_g4'
base = run_root / f'round_01/conditions/{condition}'
decisions = {row['candidate_id']: row for row in read_jsonl(base / 'guard_decisions/all.jsonl')}
audits = read_jsonl(base / 'pseudo_label_audit/all.jsonl')
false_accepts = [row for row in audits if row['false_accept']]
rejected = [row for row in decisions.values() if not row['accepted']]
display(pd.DataFrame(false_accepts[:20]))
display(pd.DataFrame(rejected[:20]))

## Exact materialized supervision

In [ ]:
mix_rows = []
for round_index in (1, 2):
    for condition_dir in sorted((run_root / f'round_{round_index:02d}/conditions').glob('*')):
        path = condition_dir / 'training_materialized/mix.json'
        if path.exists():
            mix_rows.append({'round': round_index, 'condition': condition_dir.name, **read_json(path)})
pd.DataFrame(mix_rows)

## Accuracy by round, condition, and evaluation set

In [ ]:
summary_path = run_root / 'summary.csv'
if summary_path.exists():
    results = pd.read_csv(summary_path)
    display(results.sort_values(['dataset', 'round', 'condition']))
else:
    print('The collector has not produced summary.csv yet.')